# Overview

This repository demonstrates how to set up, use, and test an Amazon Bedrock Agent with Code Interpreter capabilities. The project is divided into three Jupyter notebooks, each focusing on a specific aspect of the process.


## Context

This is the first notebook in the series to demonstrates how to set up and use an Amazon Bedrock Agent with Code Interpreter capabilities.

In this notebook we process open souce NYC Taxi and Limousine data to be used by our Amazon Bedrock Agent later
#### NYC TLC Trip Record Data

- **Source**: [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
- **Content**: Yellow and green taxi trip records (pickup/dropoff times, locations, fares, etc.)

#### Process
1. Download Parquet file for target date
2. Convert to CSV, reduce to <100MB
3. Upload to S3 for agent use

Note: Ensure S3 upload permissions

<h2>Prerequisites</h2>

Apart from the libraries that we will be installing, this notebook requires permissions to:

<ul>
<li>access Amazon Bedrock</li>
</ul>

If running on SageMaker Studio, you should add the following managed policies to your role:
<ul>
<li>AmazonBedrockFullAccess</li>
</ul>

<div class="alert alert-block alert-info">
<b>Note:</b> Please make sure to enable `Anthropic Claude 3.5 Sonnet` model access in Amazon Bedrock Console, as the later notebook will use Anthropic Claude 3.5 Sonnet model.
</div>

## Setup

We need to import the necessary Python libraries 

In [1]:
import os
import boto3
import logging
import requests
import pandas as pd
from io import BytesIO
from zipfile import ZipFile
from datetime import datetime, timedelta

In [2]:
# set a logger
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

In [3]:
import sagemaker

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Get the default S3 bucket
default_bucket = sagemaker_session.default_bucket()

print(f"Default S3 bucket: {default_bucket}")


[2024-11-12 10:56:21,442] p45579 {credentials.py:1278} INFO - Found credentials in shared credentials file: ~/.aws/credentials


sagemaker.config INFO - Not applying SDK defaults from location: /opt/homebrew/share/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/madhurpt/Library/Application Support/sagemaker/config.yaml


[2024-11-12 10:56:22,616] p45579 {credentials.py:1278} INFO - Found credentials in shared credentials file: ~/.aws/credentials


Default S3 bucket: sagemaker-us-east-1-015469603702


In [4]:
# constants
CSV_DATA_FILE: str = 'nyc_taxi_subset.csv'
# Bucket and prefix name where this csv file will be uploaded and used as S3 source by code interpreter
S3_BUCKET_NAME: str = default_bucket
PREFIX: str = 'code-interpreter-demo-data'
# This is the size of the file that will be uploaded to s3 and used by the agent (in MB)
DATASET_FILE_SIZE: float = 5

In [5]:
def download_nyc_taxi_data(start_date, end_date, data_types):
    base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    
    current_date = start_date
    while current_date <= end_date:
        for data_type in data_types:
            file_name = f"{data_type}_tripdata_{current_date.strftime('%Y-%m')}.parquet"
            url = base_url + file_name
            
            print(f"Downloading {file_name}...")
            
            response = requests.get(url)
            if response.status_code == 200:
                output_dir = f"nyc_taxi_data/{data_type}/{current_date.year}"
                os.makedirs(output_dir, exist_ok=True)
                
                with open(os.path.join(output_dir, file_name), 'wb') as f:
                    f.write(response.content)
                print(f"Successfully downloaded {file_name}")
            else:
                print(f"Failed to download {file_name}. Status code: {response.status_code}")
        
        current_date += timedelta(days=32)
        current_date = current_date.replace(day=1)

# Set the date range for which you want to download data
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 1, 31)

# Specify the types of data you want to download
data_types = ['yellow']

# Download the data
download_nyc_taxi_data(start_date, end_date, data_types)


Successfully downloaded yellow_tripdata_2024-01.parquet


## Prepare the large data and send it to S3 to be used by the agent

Now, we will prepare the data and upload it to S3. This is the new york taxi dataset. S3 allows for larger files (100MB) to be used by the agent for code interpretation, so we will upload a CSV file that is `99.67 MB` in size.

In [6]:
# Read the parquet file into a pandas DataFrame
nyc_taxi_df = pd.read_parquet("./nyc_taxi_data/yellow/2024/yellow_tripdata_2024-01.parquet")
nyc_taxi_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [7]:
def write_csv_with_size_limit(df: pd.DataFrame, 
                              output_file: str, 
                              size_limit_mb: float = 99):
    """
    This function writes a pandas DataFrame to a CSV file with a given limit
    in size (in MB)
    """
    try:
        chunk_size: int = 10000 
        total_rows = len(df)
        start_index: int = 0
        while start_index < total_rows:
            # Write a chunk of data
            end_index = min(start_index + chunk_size, total_rows)
            chunk = df.iloc[start_index:end_index]
            
            mode = 'w' if start_index == 0 else 'a'
            chunk.to_csv(output_file, mode=mode, header=(start_index == 0), index=False)
            
            # Check file size
            current_size_mb = os.path.getsize(output_file) / (1024 * 1024)
            
            if current_size_mb >= size_limit_mb:
                logger.info(f"Reached size limit. Current file size: {current_size_mb:.2f} MB")
                break
            
            start_index = end_index
            
        final_size_mb = os.path.getsize(output_file) / (1024 * 1024)
        logger.info(f"Final file size: {final_size_mb:.2f} MB")
        logger.info(f"Rows written: {end_index} out of {total_rows}")
    except Exception as e:
        logger.error(f"An error occurred while writing to the csv file with the limit of {size_limit_mb}: {e}")

In [8]:
write_csv_with_size_limit(nyc_taxi_df, CSV_DATA_FILE, size_limit_mb=0.1)

[2024-11-12 10:56:32,367] p45579 {3729796697.py:24} INFO - Reached size limit. Current file size: 1.00 MB
[2024-11-12 10:56:32,368] p45579 {3729796697.py:30} INFO - Final file size: 1.00 MB
[2024-11-12 10:56:32,369] p45579 {3729796697.py:31} INFO - Rows written: 10000 out of 2964624


In [9]:
df = pd.read_csv(CSV_DATA_FILE, nrows=10)

# Overwrite the existing CSV file with these 10 rows
df.to_csv(CSV_DATA_FILE, index=False)

# Check the updated file size in MB
size_in_bytes = os.path.getsize(CSV_DATA_FILE)
size_in_mb = size_in_bytes / (1024 * 1024)
logger.info(f"Size of the updated {CSV_DATA_FILE} is: {size_in_mb} MB")

[2024-11-12 10:56:32,593] p45579 {2699677727.py:9} INFO - Size of the updated nyc_taxi_subset.csv is: 0.0012521743774414062 MB


In [10]:
s3_client = boto3.client('s3')
s3_client.upload_file(CSV_DATA_FILE, S3_BUCKET_NAME, f"{PREFIX}/{os.path.basename(CSV_DATA_FILE)}")
s3_uri: str = f"s3://{S3_BUCKET_NAME}/{PREFIX}/{os.path.basename(CSV_DATA_FILE)}"
logger.info(f"File uploaded successfully. S3 URI: {s3_uri}")

[2024-11-12 10:56:32,659] p45579 {credentials.py:1278} INFO - Found credentials in shared credentials file: ~/.aws/credentials
[2024-11-12 10:56:32,906] p45579 {2159136852.py:4} INFO - File uploaded successfully. S3 URI: s3://sagemaker-us-east-1-015469603702/code-interpreter-demo-data/nyc_taxi_subset.csv


In [11]:
# Write the S3 URI to a text file
with open('s3_uri.txt', 'w') as f:
    f.write(s3_uri)

print("S3 URI has been written to s3_uri.txt")

S3 URI has been written to s3_uri.txt
